# Confronto Metriche di Validazione — Due Modelli

Questo notebook scarica da Google Drive i file `val_metrics.json` di due esperimenti e produce un confronto visivo e tabellare delle metriche sul validation set.

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/MammoDiffusion/'
else:
    print("Ambiente locale rilevato.")
    BASE_PATH = '../'

print(f"Percorso base: {BASE_PATH}")

#### Import

In [ ]:
import json
import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

#### Configurazione

Inserisci gli ID Google Drive dei due file `val_metrics.json` e le etichette da mostrare nei grafici.

In [ ]:
# --- MODIFICA QUI ---
# ID del file su Google Drive (la parte dopo /d/ nell'URL di condivisione)
MODEL_A = {
    "drive_id": "INSERISCI_ID_MODELLO_A",   # es. "1aBcDeFgHiJkLmNoPqRsTuVwXyZ"
    "label": "Modello A",                    # nome visualizzato nei grafici
    "color": "#2196F3",
}

MODEL_B = {
    "drive_id": "INSERISCI_ID_MODELLO_B",
    "label": "Modello B",
    "color": "#FF5722",
}
# --- FINE CONFIGURAZIONE ---

OUTPUT_DIR = os.path.join(BASE_PATH, 'experiments', 'confronto_metriche')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Cartella output: {OUTPUT_DIR}")

#### Download metriche da Google Drive

In [ ]:
def download_metrics(config: dict, dest_dir: str) -> dict:
    """Scarica val_metrics.json da Google Drive e lo restituisce come dict."""
    dest_path = os.path.join(dest_dir, f"val_metrics_{config['label'].replace(' ', '_')}.json")
    
    if not os.path.isfile(dest_path):
        print(f"Scarico metriche per '{config['label']}' da Google Drive...")
        gdown.download(id=config["drive_id"], output=dest_path, quiet=False)
    else:
        print(f"File già presente per '{config['label']}', salto il download.")
    
    with open(dest_path, 'r', encoding='utf-8') as f:
        return json.load(f)


metrics_a = download_metrics(MODEL_A, OUTPUT_DIR)
metrics_b = download_metrics(MODEL_B, OUTPUT_DIR)

print("\nMetriche caricate con successo.")

#### Tabella di confronto

In [ ]:
METRIC_KEYS = [
    ("auc",                "AUC"),
    ("accuracy",           "Accuracy"),
    ("precision_cancer",   "Precision (Cancro)"),
    ("recall_cancer",      "Recall (Cancro)"),
    ("f1_cancer",          "F1 (Cancro)"),
    ("optimal_threshold_youden", "Soglia ottimale (Youden)"),
]

rows = []
for key, label in METRIC_KEYS:
    val_a = metrics_a.get(key, "N/A")
    val_b = metrics_b.get(key, "N/A")
    delta = round(val_a - val_b, 4) if isinstance(val_a, (int, float)) and isinstance(val_b, (int, float)) else "N/A"
    rows.append({
        "Metrica": label,
        MODEL_A["label"]: val_a,
        MODEL_B["label"]: val_b,
        f"Delta ({MODEL_A['label']} - {MODEL_B['label']})": delta,
    })

df_compare = pd.DataFrame(rows)
df_compare = df_compare.set_index("Metrica")

# Stile: evidenzia il valore migliore (più alto) tra i due modelli per le metriche principali
highlight_keys = {"AUC", "Accuracy", "Precision (Cancro)", "Recall (Cancro)", "F1 (Cancro)"}

def highlight_best(row):
    if row.name not in highlight_keys:
        return [""] * len(row)
    try:
        a, b = float(row[MODEL_A["label"]]), float(row[MODEL_B["label"]])
        if a > b:
            return ["background-color: #d4edda", "background-color: #fff3cd", ""]
        elif b > a:
            return ["background-color: #fff3cd", "background-color: #d4edda", ""]
        else:
            return ["", "", ""]
    except (ValueError, TypeError):
        return [""] * len(row)

display(df_compare.style.apply(highlight_best, axis=1).format(precision=4))

#### Grafici di confronto

In [ ]:
PLOT_METRICS = [
    ("auc",              "AUC"),
    ("accuracy",         "Accuracy"),
    ("precision_cancer", "Precision\n(Cancro)"),
    ("recall_cancer",    "Recall\n(Cancro)"),
    ("f1_cancer",        "F1\n(Cancro)"),
]

keys   = [k for k, _ in PLOT_METRICS]
labels = [l for _, l in PLOT_METRICS]
vals_a = [metrics_a.get(k, 0) for k in keys]
vals_b = [metrics_b.get(k, 0) for k in keys]

x = np.arange(len(keys))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars_a = ax.bar(x - width / 2, vals_a, width, label=MODEL_A["label"], color=MODEL_A["color"], alpha=0.85)
bars_b = ax.bar(x + width / 2, vals_b, width, label=MODEL_B["label"], color=MODEL_B["color"], alpha=0.85)

for bar in bars_a:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=9)
for bar in bars_b:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Valore metrica", fontsize=12)
ax.set_title("Confronto metriche — Validation Set", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.6)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, 'confronto_metriche_barplot.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Grafico salvato in: {fig_path}")

#### Radar chart (Spider plot)

In [ ]:
RADAR_METRICS = [
    ("auc",              "AUC"),
    ("accuracy",         "Accuracy"),
    ("precision_cancer", "Precision"),
    ("recall_cancer",    "Recall"),
    ("f1_cancer",        "F1"),
]

r_keys  = [k for k, _ in RADAR_METRICS]
r_labels = [l for _, l in RADAR_METRICS]
N = len(r_labels)

vals_a_r = [metrics_a.get(k, 0) for k in r_keys]
vals_b_r = [metrics_b.get(k, 0) for k in r_keys]

# Chiudi il cerchio
vals_a_r += vals_a_r[:1]
vals_b_r += vals_b_r[:1]

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

ax.plot(angles, vals_a_r, color=MODEL_A["color"], linewidth=2, label=MODEL_A["label"])
ax.fill(angles, vals_a_r, color=MODEL_A["color"], alpha=0.15)

ax.plot(angles, vals_b_r, color=MODEL_B["color"], linewidth=2, label=MODEL_B["label"])
ax.fill(angles, vals_b_r, color=MODEL_B["color"], alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(r_labels, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8)
ax.set_title("Radar Chart — Validation Set", fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
radar_path = os.path.join(OUTPUT_DIR, 'confronto_metriche_radar.png')
fig.savefig(radar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Radar chart salvato in: {radar_path}")

#### Classification Report completo

In [ ]:
for cfg, metrics in [(MODEL_A, metrics_a), (MODEL_B, metrics_b)]:
    print("=" * 60)
    print(f"  {cfg['label']}  —  {metrics.get('experiment_name', 'N/A')}")
    print(f"  Backbone: {metrics.get('backbone', 'N/A')}")
    print(f"  Soglia (Youden): {metrics.get('optimal_threshold_youden', 'N/A')}")
    print("=" * 60)
    print(metrics.get("classification_report", "N/A"))
    print()

#### Riepilogo testuale

In [ ]:
winner_map = {}
for key, label in METRIC_KEYS[:5]:  # solo le 5 metriche principali
    a, b = metrics_a.get(key), metrics_b.get(key)
    if isinstance(a, float) and isinstance(b, float):
        if a > b:
            winner_map[label] = MODEL_A["label"]
        elif b > a:
            winner_map[label] = MODEL_B["label"]
        else:
            winner_map[label] = "Pari"

print("Vincitore per metrica:")
for metric, winner in winner_map.items():
    print(f"  {metric:30s} → {winner}")

wins_a = sum(1 for w in winner_map.values() if w == MODEL_A["label"])
wins_b = sum(1 for w in winner_map.values() if w == MODEL_B["label"])
print(f"\nVittorie totali: {MODEL_A['label']} = {wins_a} | {MODEL_B['label']} = {wins_b}")